In [ ]:
import importlib

import altair as alt
import mlflow.pyfunc
import pandas as pd
import scipy.stats

from sportsml.graph.graph import create_complete_graph
from sportsml.inference import long_to_wide


In [ ]:
SPORT = "cbb"

features_module = importlib.import_module(f"sportsml.{SPORT}.data.features")
team_name_map = importlib.import_module(f"sportsml.{SPORT}.data.nodes").team_name_map

In [ ]:
predictor = mlflow.pyfunc.load_model(f'../models/{SPORT}/pyg')

In [ ]:
gp = create_complete_graph(len(team_name_map))
model_input = pd.DataFrame({
    'team': gp.edge_index[1],
    'opp': gp.edge_index[0],
})

In [ ]:
preds = predictor.predict(model_input)

In [ ]:
team_avg = preds.groupby('team')[['preds', 'prob']].mean()
team_avg.index = team_avg.index.map(team_name_map)

In [ ]:
alt.Chart(team_avg.reset_index()).mark_circle().encode(
    x="preds", y="prob", tooltip=["team"]
)

In [ ]:
long_to_wide(preds, team_name_map=team_name_map)